# Why Heat? Solving Common Big Data Frustrations

If you've ever worked with large datasets in Python, you've likely encountered some common, frustrating roadblocks. This tutorial is designed to show you how Heat addresses these challenges directly.

### Are You Tired Of...

* **Slow I/O?** Waiting forever for your massive dataset to load from disk?
* **Manual Chunking?** Constantly needing to fine-tune `dask` chunk sizes to balance memory usage and performance?
* **Complex Setups?** Having to manually configure a Dask cluster or write `mpi4py` communication code just to perform a simple operation like a global mean?

These issues can slow down your research and add unnecessary complexity to your code. 

### The Heat Philosophy: Simple, Data-Parallel Computing

Heat is built on a **data-parallel** model. This means that you work with a single, global `dndarray` object, and Heat handles the distribution and communication for you under the hood. 

**What this means for you:**
- **No Manual Chunking:** You specify a `split` axis, and Heat automatically distributes the data as evenly as possible across all your available resources (nodes and GPUs). No more guesswork.
- **Transparent MPI:** Heat uses high-performance MPI for communication, but you rarely have to think about it. Operations like `ht.mean()` automatically perform the necessary `Allreduce` operations without you writing any MPI code.
- **Fast, Parallel I/O:** Heat's file loaders are designed to read data in parallel, directly into the memory of each process.

Let's see a quick example.

In [ ]:
from ipyparallel import Client
rc = Client(profile="default")
if not rc.ids:
    print("No engines found. Please start an ipcluster.")

### Example: A Global Mean in 3 Lines of Code

Here, we'll load the `digits` dataset (a small stand-in for a much larger dataset), distribute it across our processes, and calculate the global mean of all its features. 

In [ ]:
%%px
import heat as ht
import sklearn.datasets

# 1. Load data and distribute it across all processes along the sample axis (split=0)
X, _ = sklearn.datasets.load_digits(return_X_y=True)
X_ht = ht.array(X, split=0)

# 2. Perform a global statistical operation. Heat handles the MPI communication.
global_mean = ht.mean(X_ht)

# 3. Print the result from just one process
if X_ht.comm.rank == 0:
    print(f"Global mean of all features: {global_mean:.4f}")

[stdout:0] Global mean of all features: 7.9693


### Notice What You *Didn't* Have to Do:

- You **didn't** set a chunk size.
- You **didn't** write any `MPI.Allreduce` code.
- You **didn't** set up a Dask client or scheduler.

You just told Heat how to split the data, and it handled the rest. This simplicity is the core of Heat's design. 

Ready to learn how it works? Let's dive into the `dndarray` in the [next notebook](2_basics.ipynb).